# Monte Carlo VaR and Expected Shortfall

**Author:** Kiran  
**Last updated:** June 2026

This notebook estimates single-asset market risk for Apple (AAPL) using three standard approaches:

- **Historical** VaR and CVaR (Expected Shortfall)
- **Parametric** VaR and CVaR (variance–covariance)
- **Monte Carlo** VaR and CVaR (Geometric Brownian Motion)

Daily adjusted prices are pulled from Yahoo Finance. Returns are analyzed for mean, volatility, skewness, and tail behavior before risk metrics are computed at the 95% confidence level.

## Objective

Quantify downside risk for a long equity position by answering:

1. What is the maximum 1-day loss not exceeded with 95% probability? (**VaR**)
2. What is the average loss in the worst 5% of outcomes? (**CVaR / Expected Shortfall**)

## Workflow

1. Import libraries and configure plots
2. Download market data and compute returns
3. Analyze the return distribution
4. Compute historical VaR and CVaR
5. Compute parametric VaR and CVaR
6. Scale VaR across holding periods
7. Run Monte Carlo simulations and compare results

## 1. Import the libraries

Import the libraries:

- yfinance (yf)
- pandas (pd)
- numpy (np)
- matplotlib.pyplot (plt)
- datetime (dt)

In [ ]:
import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Run this before importing yfinance in Google Colab
# !pip install yfinance


In [ ]:
# Import the libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt


In [ ]:
# Settings the figsize parameter for the plots in this notebook to standardize the size of plots
plt.rcParams["figure.figsize"] = (15, 8)




---



## 2. Analyze the returns distribution

### Step 1

In order to calculate returns, you'll first need stock price data. For this, you can fetch data from Yahoo Finance using `yfinance`.

Follow these steps:
1.   Set the parameters for start and end date, and the ticker.
2.   Import the data from Yahoo Finance using the `yfinance` library.
3.   Print a message saying '< n > number of records downloaded'.



In [ ]:
# Set the start and end parameters
end = dt.datetime.now()
start = dt.date(end.year - 4, end.month, end.day)

# Set the ticker
ticker = 'AAPL'

# Import the data
df = yf.download(ticker, start, end, auto_adjust=False)
df.head()


### Step 2

Now that you've downloaded the data, you need to compute the simple daily returns.

Follow these steps:
1. Compute simple returns
2. Check the first five rows of the dataframe.
3. Check the last five rows of the dataframe.

In [ ]:
# Compute simple daily returns
df['daily_returns'] = df['Adj Close'].pct_change()

# Check the first and last five rows
display(df.head())
display(df.tail())


### Step 3

Let's now visualize the return distribution. For this, you will use the `hist` function from the `matplotlib.pyplot` module.
However, as we saw earlier, the first row of the returns contains null values. So we need to drop these before passing it to the `hist` function.
Setting bins equal to a large number will spread out your plot, but a low number will cause a lack of resolution.

In [ ]:
# Drop the null values from the returns series
df = df.dropna()

# Plot the return distribution
plt.hist(df.daily_returns, bins=75)
plt.title('Distribution of Daily Returns')
plt.xlabel('Daily Returns')
plt.ylabel('Frequency')
plt.show()


### Step 4

Now, you need to compute the mean and standard deviation of the returns. You also need to compute the annualized average returns using the formula below:

$\text{Average Annualized Return} = ( ( 1 + \mu ) ^ {252}) - 1$


Standard deviation for T time periods can be computed using the following formula:

$\sigma_{annual} = \sigma_{daily} * \sqrt{T}$

Follow these steps:
1. Compute the average daily returns and the annualized returns.
2. Compute the standard deviation of the returns and the annualized volatility.
3. Compute the annualized variance.
4. Compute the skewness and kurtosis of the returns.

In [ ]:
# Daily and annualized mean returns
av_daily_rets = np.mean(df.daily_returns)
av_annual_rets = (1 + av_daily_rets) ** 252 - 1

# Daily and annualized volatility
std_daily = np.std(df.daily_returns)
std_annualized = std_daily * np.sqrt(252)

# Daily and annualized variance
daily_variance = std_daily ** 2
annualized_variance = std_annualized ** 2

print(f"The daily mean return is {av_daily_rets} and the annualized mean return is {av_annual_rets}.")
print(f"The daily volatility is {std_daily} and the annualized volatility is {std_annualized}.")
print(f"The daily variance is {daily_variance} and the annualized variance is {annualized_variance}.")


Compute the skewness and excess kurtosis of returns using the **skew()** and **kurtosis()** functions from the **scipy.stats** library. 
Note: You need to add the necessary imports here.

In [ ]:
# Import the necessary functions
from scipy.stats import skew, kurtosis

# Compute skewness and excess kurtosis
skewness = skew(df.daily_returns)
excess_kurtosis = kurtosis(df.daily_returns)

print(f"The skewness of the daily returns is {skewness}.")
print(f"The excess kurtosis of the daily returns is {excess_kurtosis}.")


### Step 5

Check the normality of the stock returns distribution using the **Shapiro-Wilk test**. You can use the `shapiro()` function from the `scipy.stats` library.

The function will return two values- the first value is the t-stat of the test, and the second value is the p-value. You can use the p-value to assess the normality of the data. If the p-value is less than or equal to 0.05, you can  reject the null hypothesis of normality and assume that the data are non-normally distributed.

In [ ]:
# Import the shapiro function from the scipy.stats library
from scipy.stats import shapiro

# Compute the p_value by running the shapiro function on the returns column
p_value = shapiro(df.daily_returns)[1]

# Print the results
if p_value <= 0.05:
    print("The returns are not normally distributed.")
else:
    print("The returns are normally distributed.")




---


## 3. Historical VaR and C-VaR (Expected Shortfall)

Value at Risk (VaR) is the maximum loss that one will not exceed with a certain probability α within a given time horizon. It is given as a threshold with a given confidence level that losses will not exceed at that level.

Conditional Value at Risk (CVaR), or Expected Shortfall, is an estimate of
expected losses sustained in the worst (1 - x)% of scenarios.

### Step 1
1. Define the parameter for the confidence level for the VaR (say, 95).
2. Define the significance level (1 - confidence level)
3. Compute the historical VaR based on the significance level (1 - confidence_level).
4. Compute the historical CVaR based on the significance level (1 - confidence_level).

In [ ]:
# Define the var level parameter
var_level = 95

# Convert returns to percentage terms
rets_percent = df.daily_returns * 100

# Compute and print the historical VaR
var_95 = np.percentile(rets_percent, 100 - var_level)
print(f"The historical VaR(95) is {var_95}.")

# Sort the returns for plotting
sorted_rets = np.sort(rets_percent)

# Plot the probability of each sorted return quantile
plt.hist(sorted_rets, density=True, stacked=True)

# Draw a vertical line in the plot for the VaR 95 quantile
plt.axvline(x=var_95, color='r', linestyle='-', label='VaR 95: {0:.2f}%'.format(var_95))
plt.legend()
plt.show()


### Step 2

Compute the Expected Shortfall (CVaR) and plot the results.

In [ ]:
# Compute and print the expected shortfall
cvar_95 = rets_percent[rets_percent < var_95].mean()
print(f"The historical CVaR(95) is {cvar_95}.")

# Sort the returns for plotting
sorted_rets = np.sort(rets_percent)

# Plot the probability of each sorted return quantile
plt.hist(sorted_rets, density=True, stacked=True)

# Draw vertical lines in the plot for the VaR 95 and CVaR quantiles
plt.axvline(x=var_95, color='r', linestyle='-', label='VaR 95: {0:.2f}%'.format(var_95))
plt.axvline(x=cvar_95, color='b', linestyle='-', label='CVaR 95: {0:.2f}%'.format(cvar_95))
plt.legend()
plt.show()




---



## 4. Parametric VaR and C-VaR (Expected Shortfall)


The **parametric method VAR** (also known as **Variance/Covariance VAR**) calculation is another commonly used form of VaR calculation. This method allows you to simulate a range of possibilities based on historical return distribution properties rather than actual return values.


Use `norm.ppf()` from `scipy.stats` with the sample mean and standard deviation of returns.

In [ ]:
# Import the necessary library
from scipy.stats import norm

# Set the confidence level for VaR(95)
var_level = 95

# Set the significance level
significance_level = (100 - var_level) / 100

# Calculate the parametric VaR(95)
mu = np.mean(rets_percent)
vol = np.std(rets_percent)
pvar_95 = norm.ppf(significance_level, mu, vol)
print(f"The parametric VaR(95) is {pvar_95}.")

# Calculate the parametric CVaR(95)
pcvar_95 = rets_percent[rets_percent < pvar_95].mean()
print(f"The parametric CVaR(95) is {pcvar_95}.")


---



## 5. Scaling VaR over time

The VaR calculated in the previous sections is simply the value at risk for a single day. To estimate the VaR for a longer time duration, scale the value by the square root of time, similar to scaling volatility.

The formula for this is:

 $\text{VaR}_{\text{t days}} = \text{VaR}_{\text{1 day}} * \sqrt{t}$


 Using the above formula, let us see how VaR increases over the time for a period of an year.

 Follow these steps:

 1. Create an empty 2-d array of shape 252x2.
 2. In a for loop, iterate through all the values of days (1-252) and add the time to the first column of the array.
 3. Add the value of VaR for that time period to the second column of the array.
 4. Plot the results by passing the array to the function plot_var() defined below.

In [ ]:
def plot_var(array):
    d = pd.DataFrame(abs(array))
    d[1].plot(xlabel="Time", ylabel="Forecasted VaR-95", title="Time scaled VaR")
    plt.show()


In [ ]:
# Create an empty array to contain the VaR values
VaR_arr = np.empty([252, 2])

# Loop through the time period
for i in range(252):
    VaR_arr[i, 0] = i + 1
    VaR_arr[i, 1] = var_95 * np.sqrt(i + 1)

# Plot the results
plot_var(VaR_arr)


## 6. Monte Carlo simulations

Follow these steps:

1. Set the seed for the random number generator so that our results are reproducible.
2. Compute the log returns.
3. Compute the mean, variance, and standard deviation of the log returns.

In [ ]:
# Set the seed for reproducibility
np.random.seed(2022)

# Compute the log returns
df['log_returns'] = np.log(df['Adj Close'] / df['Adj Close'].shift(1))
log_returns = df.log_returns.dropna()

# Compute the mean, variance, and standard deviation of log returns
log_mean = log_returns.mean()
log_variance = log_returns.var()
log_std = log_returns.std()


4. Compute the drift.

For GBM price simulation, use $$\text{drift} = \mu - \frac{1}{2}\sigma^2$$, where $\mu$ is the mean of log returns and $\sigma^2$ is the variance of log returns. If you simulate log returns directly, use $\mu$ and $\sigma$ without subtracting $\frac{1}{2}\sigma^2$ again.

5. Initialize the following parameters for simulations
- n_days: the number of days
- n_sims: the number of simulations. Here, we will run 1000 simulations.
6. Compute the daily returns using random normal draws and the drift above.
7. Compute VaR(95).
8. Compute CVaR(95).


In [ ]:
# Compute the drift
drift = log_mean - (0.5 * log_variance)

# Initialize the simulation parameters
n_days = 252
n_sims = 1000

# Compute the daily returns using random normal draws
random_values = np.random.normal(0, 1, (n_days, n_sims))
daily_returns = np.exp(drift + log_std * random_values) - 1

# Compute the Monte Carlo VaR and CVaR
simulated_returns = daily_returns[-1] * 100
mc_var_95 = np.percentile(simulated_returns, 5)
mc_cvar_95 = simulated_returns[simulated_returns < mc_var_95].mean()

print(f"The Monte Carlo VaR(95) is {mc_var_95}.")
print(f"The Monte Carlo CVaR(95) is {mc_cvar_95}.")


### Simulate price paths using Monte Carlo simulations.

1. Download the Apple stock data
2. Set s0 as the value of the last adjusted Apple close price.
3. In a for loop for 100 days:
    1. Simulate normally distributed random returns.
    2. Compute the forecasted_values as s0 times the cumulative returns.
    4. Plot the simulated prices

In [ ]:
# Download the Apple stock data
df1 = yf.download(ticker, start, end, auto_adjust=False)
prices = df1['Adj Close']

# Set s0 as the last adjusted close price
s0 = float(prices.iloc[-1])

# Simulate price paths using Monte Carlo simulations
T = 100
for _ in range(100):
    rand_rets = np.random.normal(drift, log_std, T)
    forecasted_values = s0 * np.exp(np.cumsum(rand_rets))
    plt.plot(range(T), forecasted_values)

# Show the simulations
plt.title("Monte Carlo Simulated Price Paths")
plt.xlabel("Time")
plt.ylabel("Price")
plt.show()
